In [1]:
from collections import Counter, defaultdict
from pathlib import Path
from zipfile import ZipFile
import xml.etree.ElementTree as ET
from pyproj import Transformer
from shapely.geometry import LineString, Point

ns = {'k': 'http://www.opengis.net/kml/2.2'}
projection = Transformer.from_crs('EPSG:4326', 'EPSG:32639', always_xy=True)


def read_layer(filename, layer_name):
    root = ET.fromstring(ZipFile(filename).read('doc.kml'))
    folder = root.find(f".//k:Folder[k:name='{layer_name}']", ns)
    records = []
    for placemark in folder.findall('k:Placemark', ns):
        name = placemark.findtext('k:name', namespaces=ns)
        text = placemark.findtext('.//k:coordinates', default='', namespaces=ns)
        points = []
        for item in text.split():
            lon, lat = item.split(',')[:2]
            points.append((float(lon), float(lat)))
        records.append((name, points))
    return records

In [2]:
for data_dir in [Path('.'), Path('data'), Path('../data')]:
    if (data_dir / 'mpv-v-mps.-son.kmz').exists():
        break

lanes = []
for street, points in read_layer(data_dir / 'movcud-zolaqlar-velo.kmz', 'Mövcud zolaqlar'):
    if len(points) >= 2:  # 8 Noyabr pr. qeydlərindən biri boşdur
        line = LineString([projection.transform(lon, lat) for lon, lat in points])
        lanes.append((street, line))

parking = []
for name, points in read_layer(data_dir / 'mpv-v-mps.-son.kmz', 'velosiped məntəqələri yeni'):
    lon, lat = points[0]
    point = Point(projection.transform(lon, lat))
    parking.append({'name': name, 'lon': lon, 'lat': lat, 'point': point})

assert len(parking) == 185

In [3]:
street_lengths = defaultdict(float)
for street, line in lanes:
    street_lengths[street] += line.length

total_km = sum(street_lengths.values()) / 1000
longest_street = max(street_lengths, key=street_lengths.get)
print(f'{total_km:.2f}, {longest_street}')

57.14, 8 Noyabr pr.


In [4]:
for place in parking:
    distances = [(place['point'].distance(line), street) for street, line in lanes]
    place['distance'], place['street'] = min(distances)

print(len([place for place in parking if place['distance'] <= 50]))

87


In [5]:
outside = [place['name'] for place in parking
           if not (40.30 <= place['lat'] <= 40.46 and 49.70 <= place['lon'] <= 50.05)]
print(f'{len(outside)}:', ', '.join(outside))

4: MPV-144-2, MPV-145-2, MPV-146-2, MPV-147-2


In [6]:
lonely = [place for place in parking
          if place['distance'] > 50 and place['name'] not in outside]
streets = Counter(place['street'] for place in lonely)
print(len(lonely), streets.most_common(3))

vaqif = [line for street, line in lanes if street == 'Vaqif pr.'][0]
vaqif_north = max(y for x, y in vaqif.coords)
north = [place for place in lonely
         if place['street'] == 'Vaqif pr.' and place['point'].y > vaqif_north]
print(len(north))

94 [('Vaqif pr.', 27), ('Mətbuat pr.', 14), ('Xətai pr.', 13)]
25


Azadlıq prospekti üzrə, Vaqif prospektindən şimala yeni zolaq salınmasını təklif edirəm. 

Şəhərdəki 94 tənha parkinqin 27-sinin ən yaxın zolağı Vaqif prospektindədir, onların da 25-i bu zolaqdan şimalda yerləşir. 

Tikintiyə buradan başlamaq, yaxın parkinqləri qısa bağlantılarla qoşmaq ilk mərhələni sürətləndirə bilər.